# SP2013 Fraction Alignment Summary

This notebook reproduces the 1x3 SP2013 fraction alignment figure used in the NeurIPS 2026 draft. The first panel shows MAG to the human eight-cell accuracy profile. The second and third panels are intentionally blank result slots for process alignment and human-likeness preference.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

CELL_ORDER = ["add ED", "add UD", "sub ED", "sub UD", "mul ED", "mul UD", "div ED", "div UD"]
OURS_LABEL = "Ours"
OURS_MAG = 5.70
OURS_ROLLOUT = (
    "results/transformer_replication/"
    "smol2_135m_synth_unique_seed1_all1000_mincols_paramsalways_pretrained_matchspval_iduma_lr5em4_lin_e5_20260409_104738/"
    "sp2013_all1000_roll1_true_last_temp1p3/sp2013_panel25_rollouts.csv.gz"
)

notebook_path = Path.cwd()
repo_root = next(
    parent
    for parent in [notebook_path, *notebook_path.parents]
    if (parent / "sp2013_human.csv").exists() and (parent / "fractionGPT").exists()
)
paper_dir = repo_root / "writing" / "LLM_Student" / "neurips2026"
fig_dir = paper_dir / "figures"
table_dir = paper_dir / "tables"
fig_dir.mkdir(parents=True, exist_ok=True)
table_dir.mkdir(parents=True, exist_ok=True)

def cell_from_prob(prob):
    p = str(prob).replace(" ", "")
    for op, opname in [("+", "add"), ("-", "sub"), ("*", "mul"), (":", "div"), ("/", "div")]:
        if op in p:
            left, right = p.split(op, 1)
            left_den = left.split("/")[1]
            right_den = right.split("/")[1].replace("=?", "").replace("=", "")
            denom = "ED" if left_den == right_den else "UD"
            return f"{opname} {denom}"
    raise ValueError(f"Could not parse SP2013 problem: {prob}")

def baseline_cell(row):
    op = str(row["operation"]).strip().lower()
    op = {"add": "add", "sub": "sub", "mul": "mul", "div": "div"}[op[:3]]
    return f"{op} {row['denom_type']}"


In [ ]:
human_df = pd.read_csv(repo_root / "sp2013_human.csv")
human_df["cell"] = human_df["prob"].map(cell_from_prob)
human_profile = (100 * human_df.groupby("cell")["acc"].mean()).reindex(CELL_ORDER)

ours_df = pd.read_csv(repo_root / OURS_ROLLOUT)
ours_df["cell"] = ours_df["prob"].map(cell_from_prob)
ours_profile = (100 * ours_df.groupby("cell")["is_correct_true"].mean()).reindex(CELL_ORDER)

baseline_df = pd.read_csv(repo_root / "fractionGPT" / "llm_baselines" / "llm_outputs_sp2013_fractions.csv")
baseline_df["cell"] = baseline_df.apply(baseline_cell, axis=1)
baseline_profiles = {
    model: (100 * cur.groupby("cell")["is_correct"].mean()).reindex(CELL_ORDER)
    for model, cur in baseline_df.groupby("model", sort=True)
}

mag_rows = []
for model, profile in baseline_profiles.items():
    mag_rows.append({"model": model, "mag_vs_human": float((profile - human_profile).abs().mean())})
mag_rows.append({"model": OURS_LABEL, "mag_vs_human": OURS_MAG})

baseline_order = ["Gemini 2.5 Flash", "GPT-4.1 mini", "Claude Sonnet 4", OURS_LABEL]
mag_df = pd.DataFrame(mag_rows)
mag_df["plot_order"] = mag_df["model"].map({model: idx for idx, model in enumerate(baseline_order)})
mag_df = mag_df.sort_values("plot_order").drop(columns="plot_order").reset_index(drop=True)

profile_df = pd.DataFrame({"cell": CELL_ORDER, "Human": human_profile.values, OURS_LABEL: ours_profile.values})
for model, profile in baseline_profiles.items():
    profile_df[model] = profile.values
gap_df = pd.DataFrame({"cell": CELL_ORDER, "ours_minus_human": (ours_profile - human_profile).values})

mag_df.to_csv(table_dir / "sp2013_fraction_mag_human.csv", index=False)
profile_df.to_csv(table_dir / "sp2013_fraction_cell_accuracy_profiles.csv", index=False)
gap_df.to_csv(table_dir / "sp2013_fraction_ours_gap_by_cell.csv", index=False)

mag_df


In [ ]:
colors = {
    OURS_LABEL: "#1f77b4",
    "Claude Sonnet 4": "#8c8c8c",
    "GPT-4.1 mini": "#b07aa1",
    "Gemini 2.5 Flash": "#e15759",
}
plot_labels = [
    {
        "Gemini 2.5 Flash": "Gemini 2.5\nFlash",
        "GPT-4.1 mini": "GPT-4.1\nmini",
        "Claude Sonnet 4": "Claude\nSonnet 4",
        "Ours": "Ours",
    }[label]
    for label in mag_df["model"]
]

plt.rcParams.update(
    {
        "font.size": 12,
        "axes.titlesize": 13,
        "axes.labelsize": 12,
        "xtick.labelsize": 10.5,
        "ytick.labelsize": 10.5,
        "legend.fontsize": 10.5,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

fig, axes = plt.subplots(1, 3, figsize=(12.4, 3.55), constrained_layout=True)
bar_colors = [colors.get(model, "#9a9a9a") for model in mag_df["model"]]
axes[0].bar(np.arange(len(mag_df)), mag_df["mag_vs_human"], color=bar_colors)
axes[0].set_title("(a) MAG to human")
axes[0].set_ylabel("Mean absolute gap (pp)")
axes[0].set_xticks(np.arange(len(mag_df)), plot_labels)
axes[0].set_ylim(0, max(38, float(mag_df["mag_vs_human"].max()) + 4))
axes[0].tick_params(axis="x", pad=5)
for idx, val in enumerate(mag_df["mag_vs_human"]):
    axes[0].text(idx, val + 0.9, f"{val:.1f}", ha="center", va="bottom", fontsize=11)

for ax in axes[1:]:
    ax.axis("off")

fig_path = fig_dir / "sp2013_fraction_alignment_summary.png"
fig.savefig(fig_path, dpi=300, bbox_inches="tight")
fig_path
